Data Cleaning Process for Yahoo! Finance Data
--> Download adjusted closing price
--> Check and Delete NaN values
--> Calculate log returns
--> Inspect extreme returns
--> Save the cleaned data as CSV files

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt

/Users/aprilwang/Desktop/VaR & ES Analysis/Portfolio_VaR & ES_Analysis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
tickers = ["QQQ", "MRVL", "TSLA", "AVGO"]
historical_years = 5
end_date = dt.datetime.now()
start_date = end_date - dt.timedelta(days=historical_years * 365)
print("Start date:", start_date.date())
print("End date:", end_date.date())

Start date: 2021-11-12
End date: 2026-11-11


Step1
DownLoad adjusted closing price

In [3]:
raw_data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False,progress=False)
price_data= raw_data["Adj Close"].reindex(columns=tickers).sort_index(axis=0)
display(price_data.head())

Ticker,QQQ,MRVL,TSLA,AVGO
Date,,,,
2021-11-12,383.077606,72.180275,344.473328,51.556480
2021-11-15,382.990265,71.630188,337.796661,51.789902
2021-11-16,385.717560,72.514259,351.576660,52.064522
2021-11-17,385.921356,72.337448,363.003326,52.144165
2021-11-18,389.920044,73.820724,365.459991,52.617409


Step2.
Check and Delete NaN value

In [4]:
print(price_data.isna().sum())
price_data = price_data.dropna(axis=0, how='any')

Ticker
QQQ     0
MRVL    0
TSLA    0
AVGO    0
dtype: int64


Step3.
Calculate log returns

In [5]:
log_returns = np.log(price_data / price_data.shift(1)).dropna()
display(log_returns.head())

Ticker,QQQ,MRVL,TSLA,AVGO
Date,,,,
2021-11-15,-0.000228,-0.007650,-0.019573,0.004517
2021-11-16,0.007096,0.012267,0.039984,0.005289
2021-11-17,0.000528,-0.002441,0.031984,0.001529
2021-11-18,0.010308,0.020298,0.006745,0.009035
2021-11-19,0.005560,-0.007614,0.036432,-0.010651


Step4.
Inspect extreme returns:
Just select the top 5 log returns, manually examine whether it really happened or not

In [6]:
for ticker in tickers:
    largest_returns = (log_returns[ticker].abs().nlargest(5).index)
    print(f"\nLargest returns for {ticker}:")
    display(log_returns.loc[largest_returns,ticker].sort_values(ascending=False))


Largest returns for QQQ:


Date
2025-04-09    0.113356
2022-11-10    0.071193
2022-09-13   -0.056397
2025-04-04   -0.064122
2026-11-05   -0.067090
Name: QQQ, dtype: float64


Largest returns for MRVL:


Date
2026-06-02    0.281568
2023-05-26    0.280836
2024-12-04    0.208545
2025-01-27   -0.211978
2025-03-06   -0.220817
Name: MRVL, dtype: float64


Largest returns for TSLA:


Date
2025-04-09    0.204491
2024-10-24    0.198187
2025-06-05   -0.153849
2026-07-23   -0.156931
2025-03-10   -0.167546
Name: TSLA, dtype: float64


Largest returns for AVGO:


Date
2024-12-13    0.218594
2025-04-09    0.171118
2025-12-12   -0.121355
2026-06-04   -0.134525
2025-01-27   -0.191122
Name: AVGO, dtype: float64

Step5.
Save the cleaned data to a CSV file for future use

In [7]:
log_returns.to_csv("cleaned_log_returns.csv", index_label="Date")